[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week01.ipynb)

# Week 1: Deep Learning Overview & ML-to-Network Framing
**Introduction to Deep Learning (HIT)** &middot; Part I: Foundations

What deep learning is; framing a task as tensor inputs, model outputs, and a loss function.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Set up the PyTorch toolchain and confirm it runs.
- Frame any ML task as tensors in, a model, and a loss out.
- Run a first end-to-end training loop and read the loss curve.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Tensors live on a device
Everything is a tensor; a tensor lives on the CPU or the GPU.

In [ ]:
x = torch.randn(2, 3)
print("on cpu:", x.device, "| shape:", tuple(x.shape), "| dtype:", x.dtype)
x = x.to(device)
print("moved to:", x.device)
print("a few ops:", x.sum().item(), x.mean().item())

## 2. A neuron, then a layer
A linear layer is just w.x + b applied to every row of the input.

In [ ]:
layer = nn.Linear(in_features=3, out_features=1)
print("weight:", tuple(layer.weight.shape), "bias:", tuple(layer.bias.shape))
batch = torch.randn(4, 3)              # 4 samples, 3 features
print("output:", tuple(layer(batch).shape), "(one value per sample)")
# the layer just does this by hand:
by_hand = batch @ layer.weight.t() + layer.bias
print("matches manual w.x + b:", torch.allclose(layer(batch), by_hand))

## 3. Gradient descent BY HAND
Before autograd: fit y = 2x + 1 by computing the gradient of MSE yourself.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 1)
y = 2 * X + 1 + 0.1 * torch.randn(200, 1)

w, b = torch.zeros(1), torch.zeros(1)      # start at 0
lr = 0.1
for step in range(60):
    pred = w * X + b
    err = pred - y
    grad_w = (2 * err * X).mean()          # d MSE / d w
    grad_b = (2 * err).mean()              # d MSE / d b
    w -= lr * grad_w; b -= lr * grad_b
print(f"by-hand GD: w={w.item():.3f} (target 2.0), b={b.item():.3f} (target 1.0)")

## 4. The same loop with autograd
Now let PyTorch compute the gradients. Watch the four steps: forward, loss, backward, step.

In [ ]:
model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(60):
    opt.zero_grad()           # 1. clear old gradients
    pred = model(X)           # 2. forward
    loss = loss_fn(pred, y)   # 3. measure error
    loss.backward()           # 4. gradients (autograd)
    opt.step()                # 5. update weights
    losses.append(loss.item())

print(f"autograd: weight {model.weight.item():.3f} (target 2.0), bias {model.bias.item():.3f} (target 1.0)")
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("MSE loss"); plt.title("Training loss"); plt.show()

## 5. See the fitted line
Plot the data and the line the model learned.

In [ ]:
xs = torch.linspace(X.min(), X.max(), 50).unsqueeze(1)
with torch.no_grad():
    ys = model(xs)
plt.scatter(X, y, s=8, alpha=0.4, label="data")
plt.plot(xs, ys, "r", label="fit"); plt.legend(); plt.title("Learned y = 2x + 1"); plt.show()

**Try it live:** change `lr` to 0.01 and to 1.5 and re-run. The next cell sweeps all three at once.

In [ ]:
def train(lr, steps=60):
    torch.manual_seed(0)
    m = nn.Linear(1, 1); o = torch.optim.SGD(m.parameters(), lr=lr); f = nn.MSELoss()
    h = []
    for _ in range(steps):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    return h

for lr in [0.01, 0.1, 1.5]:
    plt.plot(train(lr), label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log)"); plt.legend()
plt.title("Too small crawls, good converges, too large diverges"); plt.show()

## 6. Framing: classification vs regression
Same machinery, different output layer and loss.

In [ ]:
xb = torch.randn(8, 4)                          # 8 samples, 4 features

# Classification: one logit per class + cross-entropy
clf = nn.Linear(4, 3)
ce = nn.CrossEntropyLoss()(clf(xb), torch.randint(0, 3, (8,)))
print("classification -> logits", tuple(clf(xb).shape), "| CE loss", round(ce.item(), 3))

# Regression: one value + MSE
reg = nn.Linear(4, 1)
mse = nn.MSELoss()(reg(xb), torch.randn(8, 1))
print("regression     -> output", tuple(reg(xb).shape), "| MSE loss", round(mse.item(), 3))

## 7. End to end: a tiny classifier
Train a 2-class model on a separable toy set and read accuracy.

In [ ]:
torch.manual_seed(0)
Xc = torch.randn(300, 2); yc = (Xc[:, 0] + Xc[:, 1] > 0).long()
clf = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
opt = torch.optim.Adam(clf.parameters(), lr=0.05); loss_fn = nn.CrossEntropyLoss()
for epoch in range(80):
    opt.zero_grad(); loss = loss_fn(clf(Xc), yc); loss.backward(); opt.step()
acc = (clf(Xc).argmax(1) == yc).float().mean()
print(f"final loss {loss.item():.3f} | accuracy {acc.item():.3f}")

### Mini-exercise
Generate `y = -3x + 2 + noise` and recover the slope and intercept with an `nn.Linear(1, 1)` trained by SGD. What learning rate and how many steps do you need?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
torch.manual_seed(1)
Xe = torch.randn(200, 1); ye = -3 * Xe + 2 + 0.1 * torch.randn(200, 1)
m = nn.Linear(1, 1); o = torch.optim.SGD(m.parameters(), lr=0.1); f = nn.MSELoss()
for _ in range(80):
    o.zero_grad(); f(m(Xe), ye).backward(); o.step()
print(f"recovered slope {m.weight.item():.3f} (target -3.0), intercept {m.bias.item():.3f} (target 2.0)")

**Recap:** every task is data + model + loss + optimization. The output layer and loss are chosen together (regression: 1 unit + MSE; k-class: k logits + cross-entropy). Autograd computes the gradients the hand-written loop computed manually.

---
Student materials for this week: the lab handout (`labs/week01.html`) and the curated references (`references/week01.html`) in the course site.